<a href="https://colab.research.google.com/github/vencov/FAV_course/blob/main/TutOEME/TaskOE_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task OE: Forward Pressure Level (FPL) Calibration and Ear-Canal Area Reconstruction

## Overview

In this notebook you will process real chirp-response recordings from an OAE probe and
implement the **Forward Pressure Level (FPL) calibration** method (Scheperle et al., 2008;
Souza et al., 2014), together with the **ear-canal area-function reconstruction** method of
Rasetshwane & Neely (2011, *JASA* 130:3873).

You will work with **three recordings**:

1. **A known reference tube** — a simple uniform tube of known length and diameter, so you
   have a ground-truth answer to check your implementation against.
2. **A long, lossy tube** — a tube that is long enough such thet visco-thermal losses reduced all reflected waves and the microphone therefore records the outgoing wave only - this tube response can be used to normalized other chirp responses to remove the characteristics of the speaker and improve visualizaiton of the recorded responses.
3. **A real ear canal** — recorded in the ear of volunteering students.

## Task 1 — Load your recordings

Load all the recordings you have from the lab (reference tube, lossy tube, ear canals) using the loading and pre-processing pipeline in this notebook (`load_recording`, `process_recording`,
etc.), so that you end up with a `recordings` list and a matching `processed` list, one entry per recording, in the same order.

Give each recording a clear, descriptive `name` (e.g. `"reference_tube"`, `"lossy_tube"`,`"ear_canal"`) — you'll use these names to label your plots later.

# Time-domain averaging of chirp-evoked ear-canal responses

When a chirp stimulus is repeated many times and the ear-canal recording
is cut into one epoch per chirp, the **stimulus-evoked response** is the
same (phase-locked) in every epoch, while the **noise** (physiological,
electronic) is essentially independent from repeat to repeat. Averaging
the epochs together cancels the noise (its amplitude shrinks roughly as
1/√N) while leaving the evoked response untouched — this is the standard
trick used to pull small otoacoustic/ear-canal responses out of a noisy
recording.

This notebook:
1. loads a recording consisting of `Nchirps` repeated chirps back to back,
2. cuts it into individual chirp-length epochs,
3. plots several raw (single-trial) epochs so you can see how noisy any
   *one* chirp response looks,
4. averages across epochs and overlays the averaged response on a single
   raw epoch, so you can see directly how much the noise is reduced.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import urllib.request
from scipy.signal import savgol_filter  # import savitzky golay filter
from scipy.signal.windows import blackman, tukey
plt.rcParams['figure.dpi'] = 100

visualizeChirp = True  # set to True to show processd response or False to not show it

# path to files with results (located on github, just check the correct folder)
base_url = "https://raw.githubusercontent.com/vencov/FAV_upsidedown/main/TutOEME/"

# list every recording you want the students to work with here
fnames = [
    # 'filename_LLT.mat',  # fill in titles of files where recorded responses are, first one must be LLT response
    # 'filname_tubeResp.mat',
    # 'filename_earL1.mat',
]


def load_recording(fname, base_url, force_redownload=True):
    """Download (if needed) and load one tube-response .mat recording
    into a plain dict with all the fields the processing step needs."""
    if force_redownload and os.path.exists(fname):
        os.remove(fname)
    if not os.path.exists(fname):
        urllib.request.urlretrieve(base_url + fname, fname)

    data = loadmat(fname)
    rec = dict(
        name=fname,
        y1all=data['y1all'][0],
        y2all=data['y2all'][0],
        fsamp=float(data['fsamp'][0][0]),
        Nsamp=int(data['buffersize'][0][0]),
        Nchirps=int(data['Nchirps'][0][0]),
        AmpChirp=data['AmpChirp'][0][0],
        latSC=int(data['lat_SC'][0][0]),
        MicGain=int(data['MicGain'][0][0]),
        Psrc1=data['Psrc1'][0]/2,
        Psrc2=data['Psrc2'][0]/2,
        Zsrc1=data['Zsrc1'][0],
        Zsrc2=data['Zsrc2'][0],
        fxTS=data['fxTS'][0],
    )
    return rec

recordings = [load_recording(fname, base_url) for fname in fnames]

for rec in recordings:
    print(f"Loaded '{rec['name']}': Nchirps={rec['Nchirps']}, Nsamp={rec['Nsamp']}, fsamp={rec['fsamp']} Hz")

def epoch_chirps(y, latSC, Nsamp, Nchirps):
    '''Strip the initial latency and cut y into Nchirps epochs of Nsamp samples each.'''
    y_stripped = y[latSC:]
    return np.reshape(y_stripped[:Nchirps * Nsamp], (Nchirps, Nsamp))



def reject_and_average(epochs, threshold=0.01):
    '''
    epochs: array (Nframes, Nsamp)
    Returns the artifact-free average, plus which epochs were rejected.
    '''
    mean_epoch = np.mean(epochs, axis=0)
    deviation = epochs - mean_epoch

    # max deviation WITHIN each frame -> one value per frame
    max_dev_per_frame = np.max(np.abs(deviation), axis=1)

    bad_frames = np.where(max_dev_per_frame > threshold)[0]
    good_frames = np.setdiff1d(np.arange(epochs.shape[0]), bad_frames)

    y_avg = np.mean(epochs[good_frames, :], axis=0)
    info = dict(n_total=epochs.shape[0], n_rejected=len(bad_frames), bad_frames=bad_frames)
    return y_avg, info


def process_channel(y, latSC, Nsamp, Nchirps, Nchskip=10, threshold=0.01):
    '''Epoch -> skip onset chirps -> artifact reject -> average, for one channel.'''
    epochs = epoch_chirps(y, latSC, Nsamp, Nchirps)[Nchskip:]
    return reject_and_average(epochs, threshold=threshold)


def process_recording(rec, Nchskip=10, threshold=0.01):
    '''Run process_channel on both speakers of one recording dict.'''
    y1_avg, info1 = process_channel(rec['y1all'], rec['latSC'], rec['Nsamp'], rec['Nchirps'], Nchskip, threshold)
    y2_avg, info2 = process_channel(rec['y2all'], rec['latSC'], rec['Nsamp'], rec['Nchirps'], Nchskip, threshold)
    return dict(name=rec['name'], y1_avg=y1_avg, y2_avg=y2_avg, info1=info1, info2=info2)


processed = [process_recording(rec) for rec in recordings]

for p in processed:
    print(f"{p['name']}: ch1 rejected {p['info1']['n_rejected']}/{p['info1']['n_total']}, "
          f"ch2 rejected {p['info2']['n_rejected']}/{p['info2']['n_total']}")


if visualizeChirp:
    demo_idx = 1                     # pick which recording to demonstrate
    rec = recordings[demo_idx]
    proc = processed[demo_idx]

    Nchskip = 10                     # match process_recording's default
    epochs1 = epoch_chirps(rec['y1all'], rec['latSC'], rec['Nsamp'], rec['Nchirps'])[Nchskip:]
    epochs2 = epoch_chirps(rec['y2all'], rec['latSC'], rec['Nsamp'], rec['Nchirps'])[Nchskip:]
    t_ms = np.arange(rec['Nsamp']) / rec['fsamp'] * 1000

    n_train_chirps = 5               # how many back-to-back chirps to show as "the train"
    one_chirp_idx = 0                # which single epoch to compare against the average

    fig, axs = plt.subplots(2, 2, figsize=(11, 8))

    # --- row 1: raw chirp TRAIN (continuous signal, several chirps back-to-back) ---
    n_train_samples = n_train_chirps * rec['Nsamp']
    t_train_ms = np.arange(n_train_samples) / rec['fsamp'] * 1000
    train1 = rec['y1all'][rec['latSC']:rec['latSC'] + n_train_samples]
    train2 = rec['y2all'][rec['latSC']:rec['latSC'] + n_train_samples]

    axs[0, 0].plot(t_train_ms, train1, linewidth=0.7)
    axs[0, 1].plot(t_train_ms, train2, linewidth=0.7)
    for ax, ch in zip(axs[0], ['Channel 1', 'Channel 2']):
        ax.set_title(f"{rec['name']} — {ch}: raw chirp train ({n_train_chirps} chirps)")
        ax.set_xlabel('Time (ms)')
    axs[0, 0].set_ylabel('Amplitude (a.u.)')

    # --- row 2: averaging effect -- individual epochs (gray), one single epoch, and the average ---
    n_show = 8
    for idx in range(n_show):
        axs[1, 0].plot(t_ms, epochs1[idx], color='gray', alpha=0.35, linewidth=0.7)
        axs[1, 1].plot(t_ms, epochs2[idx], color='gray', alpha=0.35, linewidth=0.7)

    axs[1, 0].plot(t_ms, epochs1[one_chirp_idx], color='C1', linewidth=1.2, label='single chirp response')
    axs[1, 1].plot(t_ms, epochs2[one_chirp_idx], color='C1', linewidth=1.2, label='single chirp response')

    n_avg1 = proc['info1']['n_total'] - proc['info1']['n_rejected']
    n_avg2 = proc['info2']['n_total'] - proc['info2']['n_rejected']
    axs[1, 0].plot(t_ms, proc['y1_avg'], color='C0', linewidth=2.0, label=f'average ({n_avg1} chirps)')
    axs[1, 1].plot(t_ms, proc['y2_avg'], color='C0', linewidth=2.0, label=f'average ({n_avg2} chirps)')

    for ax, ch in zip(axs[1], ['Channel 1', 'Channel 2']):
        ax.set_title(f"{ch}: single chirp vs. average (gray = individual chirps)")
        ax.set_xlabel('Time (ms)')
        ax.legend(fontsize=8)
    axs[1, 0].set_ylabel('Amplitude (a.u.)')

    plt.tight_layout()
    plt.show()


## Task 2 — Implement the FPL pressure decomposition

Given the pressure $P_{ec}(\omega)$ measured at the entrance of the ear canal (or tube), the
reflectance $R(\omega)$ at that plane, the Thévenin source impedance $Z_{src}(\omega)$, and the
converged surge impedance $Z_0$, implement a function that computes:

- **Forward-going pressure** $P^+$ — the pressure wave traveling *into* the canal:
$$
P^+ = \; ?
$$

- **Reverse-going pressure** $P^-$ — the pressure wave reflected *back out* of the canal:
$$
P^- = \; ?
$$

- **Source reflectance** $R_{src}$ — the reflectance of the probe itself, as "seen" by waves
  coming back out of the canal:
$$
R_{src} = \; ?
$$

- **Initial forward-going pressure** $P_{ifw}$ — the pressure that would result from a single
  forward-going wave with no further reflections from the probe:
$$
P_{ifw} = \; ?
$$

Derive these from the provided PDF and papers (Scheperle et al. 2008; Souza et al. 2014;
Rasetshwane & Neely 2011), then implement:

```python
def calcPressuresFPL(Pecs, R, Zsrc, Z0):
    '''
    Decompose measured ear-canal pressure into forward, reverse, and initial
    forward-going pressure waves.

    Parameters
    ----------
    Pecs : measured pressure at the probe plane (complex, one value per frequency)
    R    : ear-canal reflectance at the probe plane (from your calibration step)
    Zsrc : Thévenin source impedance of the probe
    Z0   : converged surge impedance (from calibrate_ear_canal)

    Returns
    -------
    Pfor, Prev, Rsrc, Pifw
    '''
    Pfor = ...   # TODO
    Prev = ...   # TODO
    Rsrc = ...   # TODO
    Pifw = ...   # TODO
    return Pfor, Prev, Rsrc, Pifw
```

Run this function on all three recordings and plot $|P_{ec}|$, $|P^+|$, $|P^-|$, and
$|P_{ifw}|$ (in dB SPL, re. 20 µPa) versus frequency, as in the example figures shown earlier
in the course.

In [ ]:
import numpy as np
from scipy.signal import savgol_filter
from scipy.signal.windows import blackman
from matplotlib.ticker import ScalarFormatter
import matplotlib.pyplot as plt

rho = 1.1769e-3   # g/cm^3, air density
c   = 3.4723e4    # cm/s, speed of sound
a   = 0.8 / 2     # cm, calibration-tube radius

fsUp   = 200000


def makeChirp(f1,f2,Nsamp,fsamp):
    '''
    generate a short chirp (linearly swept sine)
    '''
    T = Nsamp/fsamp  # duration of the chirp

    tx = np.arange(0,Nsamp/fsamp,1/fsamp)  # time axis

    A = 1
    T = Nsamp/fsamp

    y = np.zeros_like(tx)
    for k in range(len(tx)):
        y[k] = A*np.sin(2*np.pi*(f1+(f2-f1)/(2*(T))*tx[k])*tx[k])

    return y


def makeChirpTrain(f1,f2,Nsamp,fsamp,Nchirps):
    '''
    generate a train of chirps (linearly swept sines)
    '''

    chirp = makeChirp(f1,f2,Nsamp,fsamp) # make one chirp
    chirptrain = np.tile(chirp,(Nchirps,))  # repeate the chirp Nchirp times

    return chirptrain, chirp


from scipy.optimize import brentq


def _reflectance_upsampled_windowed(Zec, fxPecs, Zsurge):
    '''Reflectance R(Zec, Zsurge) on the native fxPecs grid, zero-padded to fsUp/2
    with spacing matched EXACTLY to fxPecs's own spacing (not a fsOrig guess),
    then Blackman-windowed (17 kHz half-width) to suppress TDR ringing.
    Returns (R_native, R_upsampled_windowed).'''
    R = (Zec - Zsurge) / (Zec + Zsurge)

    df = np.diff(fxPecs)[0]                    # actual spacing of fxPecs, in Hz
    L = int(round((fsUp / 2) / df)) + 1         # exact length to hit that spacing up to fsUp/2
    fxUp = np.linspace(0, fsUp / 2, L)
    Rup = np.pad(R, (0, L - len(R)))

    UpIdx = np.argmax(fxUp >= 17e3)
    if not UpIdx % 2:
        UpIdx += 1
    BMWwidth = UpIdx + UpIdx - 1
    BMwin = blackman(BMWwidth)
    BMwin = BMwin[len(BMwin)//2:]
    BMwinA = np.concatenate((BMwin, np.zeros(len(Rup) - len(BMwin))))
    return R, BMwinA * Rup

def _Rtime0_of_k(k, Zec, fxPecs):
    Zsurge = k * rho * c / (np.pi * a ** 2)
    _, Rup = _reflectance_upsampled_windowed(Zec, fxPecs, Zsurge)
    Rtime = np.fft.irfft(Rup) / 2
    RtimeNeg = np.flipud(Rtime)[:len(Rtime)//2+1]
    Rtime = Rtime[:len(Rtime)//2+1] + RtimeNeg
    return Rtime[0]

def find_k_nearest_root(Zec, fxPecs, k0=1.0, step=0.02, k_min=0.05, k_max=20.0):
    '''Find the root of Rtime0_of_k nearest to the physically sensible
    starting guess k0=1 (the calibration tube's own value), by expanding
    outward and stopping at the FIRST sign change -- avoids brentq locking
    onto a spurious, far-away root in a wide fixed bracket.'''
    f = lambda kk: _Rtime0_of_k(kk, Zec, fxPecs)
    f0 = f(k0)
    k_lo, k_hi, f_lo, f_hi = k0, k0, f0, f0
    while True:
        k_hi_new = min(k_hi + step, k_max)
        if k_hi_new > k_hi:
            f_hi_new = f(k_hi_new)
            if np.sign(f_hi_new) != np.sign(f0):
                return brentq(f, k_hi, k_hi_new)
            k_hi, f_hi = k_hi_new, f_hi_new
        k_lo_new = max(k_lo - step, k_min)
        if k_lo_new < k_lo:
            f_lo_new = f(k_lo_new)
            if np.sign(f_lo_new) != np.sign(f0):
                return brentq(f, k_lo_new, k_lo)
            k_lo, f_lo = k_lo_new, f_lo_new
        if k_hi >= k_max and k_lo <= k_min:
            raise RuntimeError('no root found within [k_min, k_max]')

def calibrate_ear_canal(Zec, fxPecs, k0=1.0, step=0.02, k_bracket=(0.05, 20.0)):
    k = find_k_nearest_root(Zec, fxPecs, k0=k0, step=step, k_min=k_bracket[0], k_max=k_bracket[1])
    Zsurge = k * rho * c / (np.pi * a ** 2)
    A0 = np.pi * a ** 2 / k
    R, _ = _reflectance_upsampled_windowed(Zec, fxPecs, Zsurge)
    return R, Zsurge, A0, k

def compute_channel_pressure(y_avg, chirpIn, fsamp, AmpChirp, MicGain, fx, mic_sensitivity=0.003):
    '''FFT ratio of averaged response to input chirp -> smoothed, interpolated, converted to Pascals.'''
    Nmean = len(y_avg)
    NmeanUp = int(2**np.ceil(2+np.log2(Nmean)))
    ChResp = np.fft.rfft(y_avg, NmeanUp) / np.fft.rfft(AmpChirp*chirpIn, NmeanUp)
    fxCh = np.arange(NmeanUp)*fsamp/NmeanUp
    fxCh = fxCh[:NmeanUp//2+1]

    RespAbs = savgol_filter(np.abs(ChResp), 20, 2) # Savitzky-Golay filtering to smooth responses and reduce the effec of noise
    RespPhase = savgol_filter(np.unwrap(np.angle(ChResp)), 20, 2)
    ChRespI = np.interp(fx, fxCh, RespAbs) * np.exp(1j*np.interp(fx, fxCh, RespPhase))

    Hinear = ChRespI / (mic_sensitivity * 10**(MicGain/20))
    Pecs = Hinear * AmpChirp
    return Pecs, fxCh, ChResp


def compute_ear_canal_impedance(Pecs, Psrc, Zsrc):
    return Zsrc * Pecs / (Psrc - Pecs)


def calcPressuresFPL(Pecs, R, Zsrc, Z0):
    '''
    Decompose measured ear-canal pressure into forward, reverse, and initial
    forward-going pressure waves.

    Parameters
    ----------
    Pecs : measured pressure at the probe plane (complex, one value per frequency)
    R    : ear-canal reflectance at the probe plane (from your calibration step)
    Zsrc : Thévenin source impedance of the probe
    Z0   : converged surge impedance (from calibrate_ear_canal)

    Returns
    -------
    Pfor, Prev, Rsrc, Pifw
    '''
    # FILL HERE THE CORRECT EQUATIONS
    '''
    Pfor = ...   # TODO FORWARD PRESSURE WAVELS
    Prev = ...   # TODO REVERSE PRESSURE WAVES
    Rsrc = ...   # TODO REFLECTION COEFFICIENT
    Pifw = ...   # TODO INITIAL FORWARD PRESSURE WAVE
    '''

    return Pfor, Prev, Rsrc, Pifw



def process_recording_fpl(rec, processed, mic_sensitivity=0.003, k_bracket=(0.05, 20.0)):
    fsamp, Nsamp, Nchirps = rec['fsamp'], rec['Nsamp'], rec['Nchirps']
    AmpChirp, MicGain, fxTS = rec['AmpChirp'], rec['MicGain'], rec['fxTS']

    _, chirpIn = makeChirpTrain(0, fsamp/2, Nsamp, fsamp, Nchirps)

    Pecs1, _, _ = compute_channel_pressure(processed['y1_avg'], chirpIn, fsamp, AmpChirp, MicGain, fxTS, mic_sensitivity)
    Pecs2, _, _ = compute_channel_pressure(processed['y2_avg'], chirpIn, fsamp, AmpChirp, MicGain, fxTS, mic_sensitivity)

    Zec1 = compute_ear_canal_impedance(Pecs1, rec['Psrc1'], rec['Zsrc1'])
    Zec2 = compute_ear_canal_impedance(Pecs2, rec['Psrc2'], rec['Zsrc2'])

    # <-- was: calibrate_ear_canal(Zec1, fxTS, num_iterations)
    R1, Z01, A01, k1 = calibrate_ear_canal(Zec1, fxTS, k_bracket=k_bracket)
    R2, Z02, A02, k2 = calibrate_ear_canal(Zec2, fxTS, k_bracket=k_bracket)

    Pfor1, Prev1, Rsrc1, Pifw1 = calcPressuresFPL(Pecs1, R1, rec['Zsrc1'], Z01)
    Pfor2, Prev2, Rsrc2, Pifw2 = calcPressuresFPL(Pecs2, R2, rec['Zsrc2'], Z02)

    return dict(name=rec['name'], fxTS=fxTS,
                Pecs1=Pecs1, Pecs2=Pecs2, Zec1=Zec1, Zec2=Zec2,
                R1=R1, R2=R2, Z01=Z01, Z02=Z02, A01=A01, A02=A02,
                Pfor1=Pfor1, Prev1=Prev1, Pifw1=Pifw1,
                Pfor2=Pfor2, Prev2=Prev2, Pifw2=Pifw2)


def plot_fpl(ax, fxTS, Pecs, Pfor, Prev, Pifw, title):
    refP = np.sqrt(2)*2e-5
    ax.semilogx(fxTS/1e3, 20*np.log10(np.abs(Pecs/refP)), 'k', label="Total pressure near source")
    ax.semilogx(fxTS/1e3, 20*np.log10(np.abs(Pfor/refP)), 'r--', label="Forward wave")
    ax.semilogx(fxTS/1e3, 20*np.log10(np.abs(Prev/refP)), 'b--', label="Reverse wave")
    ax.semilogx(fxTS/1e3, 20*np.log10(np.abs(Pifw/refP)), 'r', label="Initial outgoing wave")
    ax.set_xlabel('Frequency (kHz)'); ax.set_ylabel('|P| (dB SPL)')
    ax.set_title(title); ax.set_xlim(0.3, 13); ax.legend()
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.set_xticks([0.5, 1, 2, 4, 8, 12])
    ax.set_xticklabels(['0.5', '1', '2', '4', '8', '12'])

fpl_results = []
for rec, proc in zip(recordings, processed):
    try:
        fpl_results.append(process_recording_fpl(rec, proc))
    except ValueError as e:
        print(f"skipping {rec['name']}: {e}")

# recordings[0] / fpl_results[0] is the long lossy tube (LLT) -- used only as a
# normalization reference below, not shown directly
llt = fpl_results[0]
print(f"Using '{llt['name']}' as the normalization reference (not plotted directly).")

# sanity check: normalization by direct division only makes sense if every
# recording shares the same frequency axis as the LLT
for res in fpl_results[1:]:
    if not np.allclose(res['fxTS'], llt['fxTS']):
        print(f"WARNING: '{res['name']}' has a different fxTS than '{llt['name']}' -- "
              f"normalization below will be misaligned.")


def plot_fpl_normalized(ax, fxTS, Pecs, Pfor, Prev, Pifw, Pecs_ref, Pfor_ref, Prev_ref, Pifw_ref, title):
    ax.semilogx(fxTS/1e3, 20*np.log10(np.abs(Pecs/Pecs_ref)), 'k', label="Total pressure near source")
    ax.semilogx(fxTS/1e3, 20*np.log10(np.abs(Pfor/Pecs_ref)), 'r--', label="Forward wave")
    ax.semilogx(fxTS/1e3, 20*np.log10(np.abs(Prev/Pecs_ref)), 'b--', label="Reverse wave")
    ax.semilogx(fxTS/1e3, 20*np.log10(np.abs(Pifw/Pecs_ref)), 'r', label="Initial outgoing wave")
    ax.set_xlabel('Frequency (kHz)'); ax.set_ylabel(f'|P / P_{{{llt["name"]}}}| (dB)')
    ax.set_title(title); ax.set_xlim(0.3, 13); ax.legend()
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.set_xticks([0.5, 1, 2, 4, 8, 12])
    ax.set_xticklabels(['0.5', '1', '2', '4', '8', '12'])


# ---- pass 1: raw dB SPL, skipping the LLT itself ----
for res in fpl_results[1:]:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 8))
    plot_fpl(ax1, res['fxTS'], res['Pecs1'], res['Pfor1'], res['Prev1'], res['Pifw1'],
             f"{res['name']} — first speaker")
    plot_fpl(ax2, res['fxTS'], res['Pecs2'], res['Pfor2'], res['Prev2'], res['Pifw2'],
             f"{res['name']} — second speaker")
    plt.tight_layout()
    plt.show()
    print(f"{res['name']}: A0(speaker1)={res['A01']:.4f} cm^2, A0(speaker2)={res['A02']:.4f} cm^2")

# ---- pass 2: same recordings, normalized by the LLT response ----
for res in fpl_results[1:]:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 8))
    plot_fpl_normalized(ax1, res['fxTS'], res['Pecs1'], res['Pfor1'], res['Prev1'], res['Pifw1'],
                         llt['Pecs1'], llt['Pfor1'], llt['Prev1'], llt['Pifw1'],
                         f"{res['name']} normalized by {llt['name']} — first speaker")
    plot_fpl_normalized(ax2, res['fxTS'], res['Pecs2'], res['Pfor2'], res['Prev2'], res['Pifw2'],
                         llt['Pecs2'], llt['Pfor2'], llt['Prev2'], llt['Pifw2'],
                         f"{res['name']} normalized by {llt['name']} — second speaker")
    plt.tight_layout()
    plt.show()

## Task 3 — Compare ear-canal reflectance with the literature

For each of your three recordings:

1. Plot the magnitude (and, where relevant, group delay) of the reflectance $R(\omega)$.
2. Compare your curves against published data for a similar load — e.g. average adult  ear-canal impedance/reflectance from Rasetshwane & Neely (2011, Fig. 4).

How does your **ear canal** measurement compare, in overall shape and magnitude, to the  published adult ear-canal reflectance curves?

## Task 4 — Reconstruct and compare the ear-canal area function $A(x)$

Using the layer-peeling inverse solution built earlier in the course
(`area_function_from_calibration`), reconstruct $A(x)$ for all three recordings.

1. For the **reference tube**, compare your reconstructed $A(x)$ against the tube's actual
   known cross-sectional area (from its physical diameter). This is your sanity check — if
   this doesn't come out close to flat and correct, you cannot trust the ear canal responses too.
2. For the **ear canal**, compare your reconstructed $A(x)$ against published mean ear-canal
   area functions — e.g. Johansen (1975), Stinson & Lawton (1989), Egolf et al. (1993), or
   Rasetshwane & Neely (2011, Fig. 7).

Plot all area functions together and add there published data taken from the figure in the Task descrition provided in the pdf.



In [ ]:
def _poly_reciprocal(den, M):
    recip = np.zeros(M + 1)
    recip[0] = 1.0 / den[0]
    for n in range(1, M + 1):
        s = 0.0
        upto = min(n, len(den) - 1)
        for i in range(1, upto + 1):
            s += den[i] * recip[n - i]
        recip[n] = -recip[0] * s
    return recip

def _poly_div(num, den, M):
    recip = _poly_reciprocal(den, M)
    out = np.zeros(M + 1)
    for n in range(M + 1):
        s = 0.0
        for i in range(n + 1):
            s += num[i] * recip[n - i]
        out[n] = s
    return out

def layer_peel(r, n_sections):
    '''Recover local reflection coefficients k_0...k_{n_sections-1} from a
    causal reflectance time series r (Claerbout/Sondhi-Gopinath-style
    layer peeling -- see the FPL/area-reconstruction notebook for the derivation).'''
    M = len(r) - 1
    k = np.zeros(n_sections)
    Rm = np.array(r, dtype=float).copy()
    for m in range(n_sections):
        km = Rm[0]
        k[m] = km
        num = Rm.copy(); num[0] -= km
        den = -km * Rm; den[0] += 1.0
        Mm = M - m
        Rm1_full = _poly_div(num, den, Mm)
        Rm = np.zeros(M + 1)
        Rm[:Mm] = Rm1_full[1:Mm + 1]
    return k

def k_to_area(k, A0):
    ratio = (1.0 - k) / (1.0 + k)
    return np.concatenate(([A0], A0 * np.cumprod(ratio)))


def windowed_upsampled_R(Zec, fxPecs, Z0):
    _, Rup = _reflectance_upsampled_windowed(Zec, fxPecs, Z0)
    return Rup

def compute_TDR(R_freq):
    Rtime = np.fft.irfft(R_freq) / 2
    RtimeNeg = np.flipud(Rtime)[:len(Rtime)//2+1]
    return Rtime[:len(Rtime)//2+1] + RtimeNeg

def area_function_from_calibration(Zec, fxPecs, Z0, A0, n_sections=60, k_limit=0.995):
    Rup = windowed_upsampled_R(Zec, fxPecs, Z0)
    tdr = compute_TDR(Rup)
    n_sections = min(n_sections, len(tdr) - 1)

    margin = 10
    tdr = tdr[:n_sections + margin]
    k = layer_peel(tdr, n_sections)

    # stop at the first non-physical (or near-rigid-termination) coefficient --
    # everything past this point is noise, not real structure
    bad = np.where(np.abs(k) >= k_limit)[0]
    if len(bad) > 0:
        n_valid = bad[0]
        print(f"reflection coefficient hit |k|={np.abs(k[n_valid]):.3f} at section {n_valid} "
              f"({n_valid * c/(2*fsUp)*10:.1f} mm) -- truncating there")
        k = k[:n_valid]

    A_x = k_to_area(k, A0)
    dt = 1.0 / fsUp
    dx = c * dt / 2.0
    x_mm = np.arange(len(A_x)) * dx * 10.0
    return x_mm, A_x


# ---- apply to every recording/speaker ----
dx = c / (2 * fsUp)                    # cm per section
target_length_mm = 25                  # how far down the canal you want to reconstruct
n_sections = int(round(target_length_mm / (dx * 10)))
print(f"dx = {dx*10:.3f} mm, using n_sections = {n_sections} -> reconstructing to {n_sections*dx*10:.1f} mm")
print(f"A0 (from calibration) = {res['A01']*100:.1f} mm^2   "
      f"(expected ~{np.pi*4**2:.1f} mm^2 for an 8 mm tube)")


for res in fpl_results[1:]:
    x_mm1, A1 = area_function_from_calibration(res['Zec1'], res['fxTS'], res['Z01'], res['A01'], n_sections)
    x_mm2, A2 = area_function_from_calibration(res['Zec2'], res['fxTS'], res['Z02'], res['A02'], n_sections)
    res['x_mm1'], res['A1'] = x_mm1, A1
    res['x_mm2'], res['A2'] = x_mm2, A2

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(x_mm1, A1 * 100, label='speaker 1')   # cm^2 -> mm^2
    ax.plot(x_mm2, A2 * 100, label='speaker 2')
    ax.set_xlabel('axial distance from probe (mm)')
    ax.set_ylabel('cross-sectional area (mm$^2$)')
    ax.set_title(f"{res['name']} — ear-canal area function")
    ax.legend()
    plt.tight_layout()
    plt.show()